### First Step : Clean raw data 

The dataset has been downloaded from the website ... . Some columns are completely useless (bookmakers infos) and some rows are full of missing values. The first step is to clean the dataset.

In [1]:
import numpy as np
import pandas as pd
from Data.data_cleaning import Datacleaner

In [ ]:
cleaner = Datacleaner("./Data/ligue1_2010_2025.csv")

cleaner = (cleaner.drop_useless_columns()
           .drop_rows(row_idxs=[1520,2607,2281])
           .fillNA(column='Div',value='F1')
           .fillNA(column='Time', value='00:00')
           .add_season())

cleaner.save_data("Data/Dataset_clean.csv")

In [2]:
df = pd.read_csv("./Data/Dataset_clean.csv")

print(df)


     Div        Date   Time    HomeTeam      AwayTeam  FTHG  FTAG FTR  HTHG  \
0     F1    07/08/10  00:00     Auxerre       Lorient   2.0   2.0   D   1.0   
1     F1    07/08/10  00:00        Lens         Nancy   1.0   2.0   A   0.0   
2     F1    07/08/10  00:00        Lyon        Monaco   0.0   0.0   D   0.0   
3     F1    07/08/10  00:00   Marseille          Caen   1.0   2.0   A   0.0   
4     F1    07/08/10  00:00        Nice  Valenciennes   0.0   0.0   D   0.0   
...   ..         ...    ...         ...           ...   ...   ...  ..   ...   
5589  F1  14/12/2025  14:00        Lyon      Le Havre   1.0   0.0   H   0.0   
5590  F1  14/12/2025  16:15     Auxerre         Lille   3.0   4.0   A   0.0   
5591  F1  14/12/2025  16:15        Lens          Nice   2.0   0.0   H   1.0   
5592  F1  14/12/2025  16:15  Strasbourg       Lorient   0.0   0.0   D   0.0   
5593  F1  14/12/2025  19:45   Marseille        Monaco   1.0   0.0   H   0.0   

      HTAG  ...  AST    HC   AC    HF    AF   HY   

### Second step : Classical Bradley-Terry

#### Calibrate the lambda hyperparameter using the whole dataset

In [3]:
from Data.data_processing import DataProcesser
from BradleyTerry_classical.functions import BT_calibrate
from BradleyTerry_classical.BT_classical import BradleyTerry

In [ ]:
# Process data to get the whole dataset
processer = DataProcesser("./Data/Dataset_clean.csv")

data = processer.get_data()

teams_all = data['Teams']
W_matrix_all = data['Victory Matrix']
D_matrix_all = data['Draw Matrix']


# Calibrate lambda
Lambda_Chooser = BT_calibrate(W= W_matrix_all, D= D_matrix_all, teams= teams_all)

lambda_grid = np.linspace(-3,2,30)

Lambda_Chooser.calibrate_lambda(lambda_grid=lambda_grid)

#### Train the model with the calibrated lambda (-0.24)

In [5]:
# Process data to get the wanted year (e.g., 2010-2011)
processer = DataProcesser("./Data/Dataset_clean.csv")

processer.filter_year(year='10-11')

train_df , test_df = processer.split_train_test(test_size=11, nb_teams=20)

train_data, test_data = processer.get_train_test_data()


teams = train_data['Teams']
W_matrix = train_data['Victory Matrix']
D_matrix = train_data['Draw Matrix']


# Train the model
bt_model = BradleyTerry(lambda_draw=-0.24, learning_rate=0.01, n_iterations=1000)
bt_model.fit(W=W_matrix, D=D_matrix, teams=teams)

strength = bt_model.predict_strength()

Processed 270 matches.
Processed 110 matches.
Tolerance threshold attained after 494 iterations
Model fitted successfully !


In [ ]:
result = {team: float(value) for team,value in zip(bt_model.teams,strength)}
result